# Build & inspect an energy network

The tight dev loop for the energy work:

1. edit the model code in `src/energy/`
2. rebuild from a terminal: `snakemake -c1 energy_network` (or a single product path)
3. re-run the cells below to see the result

Only reads pipeline outputs — no visualisation code leaks into `src/energy/`.

In [ ]:
import pathlib
import sys

for _candidate in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_candidate / "_helpers.py").exists():
        sys.path.insert(0, str(_candidate))
        break
import _helpers as h  # dev-only: reads pipeline outputs, not the energy package

print("repo root:", h.REPO_ROOT)
print("built products:", h.available_products())

## (Optional) preview what a build would do

Dry-run without leaving the notebook. Drop `-n` in a terminal to actually build — long builds are better run in a terminal than here.

In [ ]:
import subprocess

result = subprocess.run(
    ["snakemake", "-n", "-c1", "energy_network"],
    cwd=h.REPO_ROOT, capture_output=True, text=True,
)
print(result.stdout[-2000:] or result.stderr[-2000:])

## Pick a product and map it

In [ ]:
name = "base-mauritius"  # or an inferred-* product from h.available_products()
h.plot_network(name);

## Inspect the tables

In [ ]:
nodes, edges = h.load_layers(name)
display(nodes.head())
display(edges.head())
edges["source"].value_counts(dropna=False)

## PyPSA view

The same topology as a PyPSA network, for electrical inspection.

In [ ]:
n = h.load_pypsa(name)
print(n)
n.buses.head()

## Compare two products side by side

In [ ]:
import matplotlib.pyplot as plt

wanted = ("base-mauritius", "inferred-osm-mauritius-rodrigues")
pair = [p for p in wanted if p in h.available_products()]
cols = max(len(pair), 1)
fig, axes = plt.subplots(1, cols, figsize=(8 * cols, 9))
axes = [axes] if cols == 1 else list(axes)
for ax, nm in zip(axes, pair):
    h.plot_network(nm, ax=ax)
plt.tight_layout()